In [1]:
from pathlib import Path
import pandas as pd

STUDY = Path("Assignment 2_multiomics target discovery") / "all_phase2_target_2018_pub"
rpkm = pd.read_csv(STUDY / "data_mrna_seq_rpkm.txt", sep="\t", comment="#", low_memory=False)
mut  = pd.read_csv(STUDY / "data_mutations.txt",     sep="\t", comment="#", low_memory=False)

# 1. RNA layer: wide gene x sample matrix -> one row per gene symbol
sample_cols = [c for c in rpkm.columns if c.startswith("TARGET-")]
rna = (rpkm.dropna(subset=["Hugo_Symbol"])
           .groupby("Hugo_Symbol")[sample_cols].mean())
rna_gene = pd.DataFrame({
    "mean_rpkm":   rna.mean(axis=1),
    "median_rpkm": rna.median(axis=1),
})

# 2. Genomic layer: one row per mutation event -> one row per gene
n_samples = mut["Tumor_Sample_Barcode"].nunique()
mut_gene = pd.DataFrame({
    "n_mutated_samples": mut.groupby("Hugo_Symbol")["Tumor_Sample_Barcode"].nunique(),
})
mut_gene["mut_freq"] = mut_gene["n_mutated_samples"] / n_samples

# 3. Join on gene symbol
df = rna_gene.join(mut_gene, how="left")
df["n_mutated_samples"] = df["n_mutated_samples"].fillna(0).astype(int)
df["mut_freq"] = df["mut_freq"].fillna(0.0)
df.index.name = "gene"

print("joined table:", df.shape)
df.sort_values("mut_freq", ascending=False).head()


joined table: (23723, 4)


,mean_rpkm,median_rpkm,n_mutated_samples,mut_freq
gene,,,,
NRAS,41.687587,40.2111,16,0.106667
KRAS,18.357734,15.4633,8,0.053333
TP53,18.563865,13.0644,6,0.040000
CREBBP,47.872242,39.7419,6,0.040000
PTPN11,15.750553,15.3414,6,0.040000


### Concordance — do the genomic and transcriptomic layers point the same way?

`mut_freq` and `mean_rpkm` are both non-negative *magnitudes*, so comparing their signs would be
vacuous (every gene would come out "concordant"). Concordance needs a **direction** on each layer.

This study gives us two directional, sample-matched signals for 158 tumors:

| layer | file | direction |
|---|---|---|
| genomic | `data_cna.txt` | GISTIC call: `+1/+2` gained, `-1/-2` deleted |
| transcriptomic | `data_mrna_seq_rpkm_zscores_ref_diploid_samples.txt` | z-score vs. the diploid samples for that gene |

For each gene we ask: in the samples where the gene is copy-number **altered**, does its expression
shift in the direction the copy number predicts (relative to the copy-neutral samples)?


In [2]:
import warnings

import numpy as np

MIN_ALTERED = 5     # need this many CNA-altered samples before we trust a direction
EPS_RNA     = 0.10  # |expression shift| below this counts as "no move" (buffered)

# 4. Directional layers, on the samples measured by BOTH assays
cna = pd.read_csv(STUDY / "data_cna.txt", sep="\t", comment="#", low_memory=False)
zsc = pd.read_csv(STUDY / "data_mrna_seq_rpkm_zscores_ref_diploid_samples.txt",
                  sep="\t", comment="#", low_memory=False)

shared = sorted(set(c for c in cna.columns if c.startswith("TARGET-")) &
                set(c for c in zsc.columns if c.startswith("TARGET-")))
print(f"samples with both CNA and RNA: {len(shared)}")

C = cna.dropna(subset=["Hugo_Symbol"]).groupby("Hugo_Symbol")[shared].mean()
Z = zsc.dropna(subset=["Hugo_Symbol"]).groupby("Hugo_Symbol")[shared].mean()
genes = C.index.intersection(Z.index)
Cv, Zv = C.loc[genes].to_numpy(), Z.loc[genes].to_numpy()

# 5. Per gene: which samples are altered, and how does expression respond?
altered   = np.abs(Cv) >= 1
n_altered = altered.sum(axis=1)

# a gene altered in every sample (or in none) leaves one side of the contrast empty -> NaN
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    cna_dir = np.nanmean(np.where(altered, Cv, np.nan), axis=1)          # net gain (+) vs loss (-)
    rna_dir = (np.nanmean(np.where(altered,  Zv, np.nan), axis=1)        # altered samples ...
             - np.nanmean(np.where(~altered, Zv, np.nan), axis=1))       # ... minus copy-neutral

# 6. Sign agreement, with a dead band so near-zero moves aren't called a direction
gsign = np.sign(np.nan_to_num(cna_dir))
rsign = np.where(np.abs(np.nan_to_num(rna_dir)) < EPS_RNA, 0, np.sign(np.nan_to_num(rna_dir)))

testable    = (n_altered >= MIN_ALTERED) & np.isfinite(rna_dir) & (gsign != 0)
concordance = np.full(len(genes), "insufficient_data", dtype=object)
concordance[testable & (gsign > 0) & (rsign > 0)] = "concordant_up"     # gained  -> higher
concordance[testable & (gsign < 0) & (rsign < 0)] = "concordant_down"   # deleted -> lower
concordance[testable & (gsign * rsign < 0)]       = "discordant"        # moves the wrong way
concordance[testable & (rsign == 0)]              = "buffered"          # CNA, but RNA unmoved

conc = pd.DataFrame({"n_altered": n_altered, "cna_dir": cna_dir,
                     "rna_dir": rna_dir, "concordance": concordance}, index=genes)

# 7. Attach to the joined table from above
df = df.join(conc, how="left")
df["concordance"] = df["concordance"].fillna("insufficient_data")
# build the boolean AFTER the join, so it stays bool dtype rather than object-with-NaN
df["concordant"] = df["concordance"].str.startswith("concordant")

print(df["concordance"].value_counts().to_string())
df.loc[df["concordant"]].sort_values("cna_dir").head(10).round(3)


samples with both CNA and RNA: 158
concordance
buffered             5748
insufficient_data    5737
discordant           4975
concordant_down      3681
concordant_up        3582


,mean_rpkm,median_rpkm,n_mutated_samples,mut_freq,n_altered,cna_dir,rna_dir,concordance,concordant
gene,,,,,,,,,
BTLA,0.835,0.232,0,0.0,21.0,-1.524,-0.160,concordant_down,True
MTAP,1.310,1.134,0,0.0,63.0,-1.508,-0.279,concordant_down,True
CD200,15.692,11.482,0,0.0,22.0,-1.500,-0.281,concordant_down,True
IFNE,0.010,0.000,0,0.0,53.0,-1.396,-0.183,concordant_down,True
IFNA8,0.007,0.000,0,0.0,52.0,-1.385,-0.148,concordant_down,True
IFNA5,0.046,0.000,0,0.0,51.0,-1.373,-0.166,concordant_down,True
IFNA22P,0.011,0.000,0,0.0,51.0,-1.373,-0.165,concordant_down,True
IFNA6,0.005,0.000,0,0.0,51.0,-1.373,-0.109,concordant_down,True
IFNA2,0.284,0.000,0,0.0,51.0,-1.373,-0.116,concordant_down,True


### Multi-evidence score

**Weights.** The course default is `{transcriptomic: 1/3, proteomic: 1/3, genomic: 1/3}`. TARGET ALL
phase 2 runs no proteomic assay, so the proteomic third is dropped and its weight split evenly
across the two layers this study *does* measure. Justify this in the report.

**Per-layer magnitude.**

- **transcriptomic** — the fraction of samples where the gene is an expression outlier (`|z| > 2`
  against the diploid reference). A first attempt used `|mean z|`, which ranked keratin-associated
  and olfactory-receptor genes at the top: those sit near zero RPKM in every sample, so the diploid
  reference SD is tiny and a single stray read produces a huge z. Two guards fix it — an expression
  floor (`median_rpkm >= 1`, keeping 10,422 of 23,723 genes) and counting outlier *prevalence*
  rather than averaging a ratio whose denominator is unstable.
- **genomic** — two assays on different units (mutation frequency, GISTIC magnitude), so each is
  rank-normalized first and then averaged. The CNA side is weighted by prevalence
  (`|cna_dir| × n_altered / 158`) so a strong call in three tumors doesn't outrank a moderate one
  in sixty.


In [3]:
MIN_MEDIAN_RPKM = 1.0   # expression floor: drop genes the assay can't measure reliably
Z_OUTLIER       = 2.0   # |z| above this counts the sample as an expression outlier


# 8. Scoring helpers (from the starter notebook)
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)


# no proteomic assay in this study -> its third is split across the two measured layers
EQUAL_WEIGHTS = {"transcriptomic": 1/2, "genomic": 1/2}

# 9. Transcriptomic magnitude: how often the gene is an expression outlier
zsc_cols = [c for c in zsc.columns if c.startswith("TARGET-")]
Zall = zsc.dropna(subset=["Hugo_Symbol"]).groupby("Hugo_Symbol")[zsc_cols].mean()
Za = Zall.reindex(df.index)

df["expressed"] = df["median_rpkm"] >= MIN_MEDIAN_RPKM
df["frac_outlier"] = (Za.abs() > Z_OUTLIER).sum(axis=1) / Za.notna().sum(axis=1).replace(0, np.nan)
df["transcriptomic"] = df["frac_outlier"].fillna(0.0)

# 10. Genomic magnitude: rank-blend the two assays so neither unit dominates
df["cna_burden"] = df["cna_dir"].abs().fillna(0.0) * df["n_altered"].fillna(0.0) / len(shared)
df["genomic"] = (rank_percentile(df["mut_freq"]) + rank_percentile(df["cna_burden"])) / 2

# 11. Score the genes that clear the expression floor, then rank
scored = df.loc[df["expressed"]].copy()
scored["score"] = multi_evidence_score(scored, ["transcriptomic", "genomic"], EQUAL_WEIGHTS)
ranked = scored.sort_values("score", ascending=False)

print(f"genes scored: {len(ranked)} of {len(df)} "
      f"({len(df) - len(ranked)} below median_rpkm {MIN_MEDIAN_RPKM})")
print(ranked.head(15)[["median_rpkm", "mut_freq", "cna_burden",
                       "transcriptomic", "genomic", "concordance", "score"]].round(3).to_string())


genes scored: 10422 of 23723 (13301 below median_rpkm 1.0)
         median_rpkm  mut_freq  cna_burden  transcriptomic  genomic    concordance  score
gene                                                                                     
HLA-C         68.346     0.020       0.177           0.079    0.986  concordant_up  0.993
GOPC           4.877     0.007       0.114           0.084    0.908  concordant_up  0.992
ARID5B        15.456     0.007       0.146           0.079    0.956  concordant_up  0.992
KPNB1         30.172     0.007       0.139           0.079    0.947  concordant_up  0.991
DYRK1A        30.868     0.000       0.373           0.094    0.743  concordant_up  0.987
BACH1          1.171     0.000       0.354           0.103    0.741  concordant_up  0.986
PIGP           3.519     0.000       0.373           0.089    0.743  concordant_up  0.986
SON           91.553     0.000       0.367           0.084    0.742  concordant_up  0.984
SUMO3         45.384     0.000       0.26

### Rank, inspect, and export

`targets_leukemia.csv` is the hand-off to Week 3.

The positive-control panel is pediatric lymphoid leukemia: TP53, NOTCH1, ATM and the other
recurrently altered genes in TARGET ALL phase 2. Recovery is partial and the misses are
informative — a gene can be a textbook driver and still score low here if the evidence for it
lives in a layer this pipeline doesn't read (fusions, promoter methylation, subtype-restricted
point mutations), so the misses are written up alongside the hits.


In [4]:
REPORT_COLS = ["median_rpkm", "mut_freq", "cna_dir", "cna_burden",
               "transcriptomic", "genomic", "concordance", "concordant", "score"]

# recurrently altered genes in pediatric lymphoid leukemia, as the positive-control panel
ALL_PANEL = {
    "tumor suppressors / DNA damage": ["TP53", "ATM", "CDKN2A", "CDKN2B", "PTEN", "FBXW7"],
    "B-ALL":                          ["ETV6", "RUNX1", "PAX5", "IKZF1", "ARID5B", "CRLF2", "SH2B3"],
    "T-ALL":                          ["NOTCH1", "BCL11B", "TAL1", "LMO2", "PHF6", "LEF1", "MYB", "WT1"],
    "kinase / RAS signalling":        ["NRAS", "KRAS", "PTPN11", "FLT3", "JAK1", "JAK2", "JAK3", "IL7R"],
    "epigenetic / relapse":           ["CREBBP", "EP300", "KMT2D", "USP7", "NT5C2"],
}

# 12. Top 15 and the full ranked export
top = ranked.head(15)
ranked.to_csv("targets_leukemia.csv")
print(f"wrote targets_leukemia.csv  ({len(ranked)} genes x {ranked.shape[1]} cols)\n")
print(top[REPORT_COLS].round(3).to_string())

# 13. Positive controls — how much of the known biology did the score recover?
order = list(ranked.index)
n = len(order)
recovered = 0
for group, panel in ALL_PANEL.items():
    print(f"\n{group}")
    for gene in panel:
        if gene not in order:
            why = ("below the expression floor" if gene in df.index else "absent from the RNA matrix")
            print(f"  {gene:8s}      --  not scored ({why})")
            continue
        rank = order.index(gene) + 1
        recovered += rank <= n * 0.10
        print(f"  {gene:8s} {rank:6d}/{n}  {'top decile' if rank <= n * 0.10 else ''}")

panel_size = sum(len(v) for v in ALL_PANEL.values())
print(f"\npanel genes in the top decile: {recovered}/{panel_size}")

# 14. Discordant genes among the top hits: strong evidence, but RNA defies the copy number
disc = ranked.head(200).loc[lambda d: ~d["concordant"]]
print(f"\ndiscordant or unmoved genes in the top 200: {len(disc)}")
print(disc.head(10)[["cna_dir", "rna_dir", "n_altered",
                     "transcriptomic", "concordance", "score"]].round(3).to_string())


wrote targets_leukemia.csv  (10422 genes x 15 cols)

         median_rpkm  mut_freq  cna_dir  cna_burden  transcriptomic  genomic    concordance  concordant  score
gene                                                                                                          
HLA-C         68.346     0.020    0.848       0.177           0.079    0.986  concordant_up        True  0.993
GOPC           4.877     0.007    0.581       0.114           0.084    0.908  concordant_up        True  0.992
ARID5B        15.456     0.007    1.000       0.146           0.079    0.956  concordant_up        True  0.992
KPNB1         30.172     0.007    0.786       0.139           0.079    0.947  concordant_up        True  0.991
DYRK1A        30.868     0.000    1.135       0.373           0.094    0.743  concordant_up        True  0.987
BACH1          1.171     0.000    1.057       0.354           0.103    0.741  concordant_up        True  0.986
PIGP           3.519     0.000    1.135       0.373        